# AETHER STT — data prep notebook (run on a T4)

Builds the Mimi extraction cache (semantic codes + byte targets) for
`configs/ctc_base.yaml` and pushes it to Google Drive. This is the only
notebook that needs to download raw LibriSpeech audio (~30GB, disposable)
- run it once, on the **cheapest GPU tier available (T4 is enough)**, since
Mimi is a small model and extraction doesn't benefit from a stronger GPU.

Once this finishes, `train_ctc.ipynb` just restores the small (~100-150MB)
extracted cache from Drive in seconds - no need to re-run this notebook
unless `configs/ctc_base.yaml`'s data settings change (a mismatched
fingerprint makes `prepare_cache` raise rather than silently reusing a
stale cache, so you'll know if a re-run is needed).

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/karl4th/aether-v3.git"
REPO_NAME = "aether-v3"

# Idempotent regardless of how many times this cell is re-run in the same
# kernel session: after the first run cwd is already inside the repo (from
# os.chdir below), so checking os.path.isdir("aether-v3") relative to cwd
# would look one level too deep and clone a second copy nested inside the
# first - repeatable indefinitely. Instead, explicitly handle "already
# standing inside the repo" as its own case.
cwd = Path.cwd()
if cwd.name == REPO_NAME and (cwd / ".git").is_dir():
    subprocess.run(["git", "pull"], check=True, cwd=cwd)
    repo_dir = cwd
    print("Already inside the repo, pulled.")
else:
    repo_dir = cwd / REPO_NAME
    if (repo_dir / ".git").is_dir():
        subprocess.run(["git", "-C", str(repo_dir), "pull"], check=True)
        print("Pulled")
    elif repo_dir.exists():
        raise RuntimeError(
            f"{repo_dir} exists but isn't a git checkout (no .git/) - "
            "remove or rename it manually before re-running this cell."
        )
    else:
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
        print("Cloned")
    os.chdir(repo_dir)

print("cwd:", os.getcwd())

In [ ]:
import importlib.util
import subprocess
import sys


def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


# Most GPU notebook images already ship a CUDA-matched torch build — don't
# clobber it. Only install if genuinely missing.
if importlib.util.find_spec("torch") is None:
    pip_install("torch")
if importlib.util.find_spec("torchaudio") is None:
    pip_install("torchaudio")

pip_install(
    "transformers>=5.17",  # <5.17 lacks MimiModel.get_audio_codes_mask - see mimi_wrapper.py
    "datasets>=2.19,<4.0",  # >=4.0 requires torchcodec + system ffmpeg for Audio decoding
    "soundfile",
    "librosa",  # datasets<4.0's Audio decode path needs this alongside soundfile
    "pyyaml",
    "numpy",
    "tqdm",
)

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
print("using device:", DEVICE)
assert DEVICE == "cuda", "No GPU visible on this VM - check the runtime's accelerator (T4 is enough)."


## Google Drive cache

Mounts Drive and mirrors just the small extracted cache there (not the raw
audio) around the `prepare_cache` call below - so `train_ctc.ipynb` never
has to redo this step on a more expensive GPU tier.

In [ ]:
import os
from pathlib import Path

from aether_v3.config import load_config

real_config = load_config("configs/ctc_base.yaml")

try:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/aether-v3")
except ImportError:
    DRIVE_ROOT = Path("drive_cache").resolve()
    print(f"Not running in Colab - falling back to local '{DRIVE_ROOT}' (no cross-session persistence).")

DRIVE_CACHE_DIR = DRIVE_ROOT / "data_cache" / Path(real_config.data.cache_dir).name

# Match extraction parallelism to whatever this VM actually has.
real_config.data.extraction_num_workers = max(2, min(os.cpu_count() or 4, 16))

print("Drive cache dir:", DRIVE_CACHE_DIR)

## Extract

**This downloads the full configured splits, not a sample** — unless the
extracted cache is already on Drive, in which case this cell just restores
it and skips download+extraction entirely. On a cold cache: per
`configs/ctc_base.yaml`, `train.100` (~6GB) + `train.360` (~24GB) for
training, plus `dev-clean`/`dev-other`/`test-clean`/`test-other` (a few
hundred MB each) for eval — roughly **30-35GB of raw audio**, decoded
through Mimi and discarded; only the ~100-150MB extraction result is kept
(locally, and mirrored to Drive below).

If you want to sanity-check on less data/compute before committing to the
full 460h, edit `configs/ctc_base.yaml`'s `data.train_splits` down to just
`["clean/train.100"]` before running this cell (and delete both the local
`data_cache/ctc_base/` and its Drive mirror if you already ran it with the
larger split — a mismatched fingerprint makes `prepare_cache` raise rather
than silently reusing it).

In [ ]:
from aether_v3.data.cache_sync import hydrate_from_remote, push_to_remote
from aether_v3.data.mimi_cache import prepare_cache

restored = hydrate_from_remote(real_config.data.cache_dir, DRIVE_CACHE_DIR)
if restored:
    print(f"Restored from Drive, skipping re-extraction for: {restored}")

prepare_cache(real_config, device=DEVICE)

pushed = push_to_remote(real_config.data.cache_dir, DRIVE_CACHE_DIR)
if pushed:
    print(f"Pushed newly extracted cache to Drive: {pushed}")

## Done

The cache is now on Drive. You can disconnect this runtime and switch to
`train_ctc.ipynb` on a stronger GPU (A100/L4) for the actual training run -
it will restore this cache from Drive in seconds.